**Лабораторна робота № 2**

**Тема: **Наївний баєсів класифікатор для аналізу тональності текстів

**Мета роботи:** Навчитися реалізовувати та використовувати наївний баєсів класифікатор для аналізу тональності текстів, оцінювати його ефективність.

**Завдання:**

*1. Завантажити та попередньо обробити дані рецензій за допомогою функції movie_review.*

*2. Побудувати словник частотності слів для позитивного і негативного класів.*

*3. Обчислити логарифмічні апріорні ймовірності (log prior) для кожного класу.*

*4. Обчислити логарифмічні правдоподібності (log likelihood) для кожного слова і класу.*

*5. Реалізувати функцію наївного баєсового класифікатора.*

*6. Оцінити точність класифікатора на тестовому наборі даних.*

*7. Проаналізувати найбільш позитивні та негативні слова за співвідношенням їх частотностей.*

*8. Проаналізувати помилки класифікації.*

*9. Протестувати класифікатор на власному твіті. Розгорніть відповідну сторінку на GitHub.*

*10. Завантажити код та результати на GitHub, надати посилання на них та сторінку у Moodle.*


***1. Завантажити та попередньо обробити дані рецензій за допомогою функції movie_review.***

In [ ]:
import nltk
import numpy as np
import pandas as pd

In [ ]:
from nltk.corpus import movie_reviews
#from utils import process_reviews, build_freqs

In [ ]:
nltk.download('movie_reviews')
nltk.download('stopwords')

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
pos_fileids = movie_reviews.fileids('pos')
neg_fileids = movie_reviews.fileids('neg')

all_positive_reviews = [movie_reviews.raw(fileid) for fileid in pos_fileids]
all_negative_reviews = [movie_reviews.raw(fileid) for fileid in neg_fileids]

total_positive_lenght = len(all_positive_reviews)
total_negative_lenght = len(all_negative_reviews)

percent= 0.80

train_positive_count = round(total_positive_lenght * percent)
train_negative_count = round(total_negative_lenght * percent)

test_pos = all_positive_reviews[train_positive_count:]
train_pos = all_positive_reviews[:train_positive_count]
test_neg = all_negative_reviews[train_negative_count:]
train_neg = all_negative_reviews[:train_negative_count]

train_x = train_pos + train_neg
test_x = test_pos + test_neg

train_y = np.append(np.ones(len(train_pos)), np.zeros(len(train_neg)))
test_y = np.append(np.ones(len(test_pos)), np.zeros(len(test_neg)))

In [ ]:
import re
import string

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer

from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

def process_review(review):
    """
    Очищення тексту, токенізація, видалення стоп-слів та стемінг

    """

    stemmer = PorterStemmer()
    stopwords_english = stopwords.words('english')
    review = re.sub(r'\$\w*', '', review)
    review = re.sub(r'^RT[\s]+', '', review)
    review = re.sub(r'https?://[^\s\n\r]+', '', review)
    review = re.sub(r'#', '', review)
    tokenizer = TweetTokenizer(
        preserve_case=False, strip_handles=True, reduce_len=True
    )
    review_tokens = tokenizer.tokenize(review)

    reviews_clean = []
    for word in review_tokens:
        if (
            word not in stopwords_english
            and any(ch.isalpha() for ch in word)
        ):
            stem_word = stemmer.stem(word)
            reviews_clean.append(stem_word)

    return reviews_clean

In [ ]:
custom_review = "This movie was AMAZING!!! Best acting ever, totally loved it. Check review at http://imdb.com #cinema #mustwatch $100 budget :)"

print(process_review(custom_review))

['movi', 'amaz', 'best', 'act', 'ever', 'total', 'love', 'check', 'review', 'cinema', 'mustwatch', 'budget']


***2. Побудувати словник частотності слів для позитивного і негативного класів***

Алгоритм:
- Для кожної рецензії у тренувальному наборі:
    - Розбити рецензію на слова з допомогою process_review.
    - Для кожного слова:
        - Створити ключ - кортеж (слово, мітка класу).
        - Якщо ключ вже є в словнику, збільшити його значення на 1.
        - Інакше додати ключ до словника зі значенням 1.


In [ ]:
def count_reviews(result, reviews, ys):
    for y, review in zip(ys, reviews):
        for word in process_review(review):
            pair = (word, y)

            if pair in result:
                result[pair] += 1
            else:
                result[pair] = 1

    return result



In [ ]:
freqs = count_reviews({}, train_x, train_y)

In [ ]:
len(freqs)

43376

***3-4. Обчислити логарифмічні апріорні ймовірності (log prior) та правдоподібності (log likelihood).***

Алгоритм:
- Порахувати кількість унікальних слів V.
- Порахувати загальну кількість позитивних слів N_pos і негативних слів N_neg.
- Порахувати кількість документів D.
- Порахувати кількість позитивних документів D_pos і негативних документів D_neg.
- Обчислити log prior:
    logprior = log(D_pos) - log(D_neg)
- Для кожного слова у словнику:
    - Дістати частотності слова для позитивного і негативного класів.
    - Обчислити ймовірності слова для кожного класу з лапласівським згладжуванням:
        p_w_pos = (freq_pos + 1) / (N_pos + V)
        p_w_neg = (freq_neg + 1) / (N_neg + V)  
    - Обчислити log likelihood як логарифм відношення ймовірностей:  
        loglikelihood = log(p_w_pos / p_w_neg)


In [ ]:
def train_naive_bayes(freqs, train_x, train_y):
    loglikelihood = {}
    logprior = 0

    vocab = set([pair[0] for pair in freqs.keys()])
    V = len(vocab)

    N_pos = N_neg = 0
    for pair in freqs.keys():
        if pair[1] > 0:
            N_pos += freqs[pair]
        else:
            N_neg += freqs[pair]

    D = len(train_y)
    D_pos = sum(train_y)
    D_neg = D - D_pos

    logprior = np.log(D_pos) - np.log(D_neg)

    for word in vocab:
        freq_pos = freqs.get((word, 1), 0)
        freq_neg = freqs.get((word, 0), 0)

        p_w_pos = (freq_pos + 1) / (N_pos + V)
        p_w_neg = (freq_neg + 1) / (N_neg + V)

        loglikelihood[word] = np.log(p_w_pos) - np.log(p_w_neg)

    return logprior, loglikelihood

In [ ]:
logprior, loglikelihood = train_naive_bayes(freqs, train_x, train_y)

In [ ]:
logprior

np.float64(0.0)

In [ ]:
loglikelihood

{'life-alt': np.float64(0.5887119092279995),
 'facto': np.float64(1.281859089787945),
 'shoe': np.float64(-0.10443527133194586),
 'antennae-lik': np.float64(0.5887119092279995),
 'negat': np.float64(-1.3571982398273121),
 'no-no': np.float64(-1.203047560000055),
 'crane': np.float64(0.7428625890552567),
 'corral': np.float64(-0.7975824518918913),
 "mother'": np.float64(0.8118554605422101),
 'kashiwabara': np.float64(-0.7975824518918913),
 'yell': np.float64(-1.4654118244675463),
 'summat': np.float64(-0.10443527133194586),
 'vacanc': np.float64(1.8414748777233658),
 'perch': np.float64(0.30102983677621786),
 "ugly'": np.float64(-0.7975824518918913),
 "rapist'": np.float64(0.5887119092279995),
 'shawshank': np.float64(-0.3275788226461547),
 'bossi': np.float64(-0.7975824518918913),
 'rumor': np.float64(0.381072544449756),
 'lanc': np.float64(-0.1734281428188975),
 'badly-dub': np.float64(0.5887119092279995),
 "jolie'": np.float64(-2.669384628793482),
 'color': np.float64(0.9536856559814

***5. Реалізувати функцію наївного баєсового класифікатора (функція класифікації рецензій)***

Алгоритм:
- Обробити рецензію з допомогою process_review.
- Встановити початкову ймовірність p = logprior.  
- Для кожного слова у рецензії:
    - Якщо слово є в словнику loglikelihood:
        - Додати loglikelihood цього слова до p.
- Повернути суму ймовірностей p.


In [ ]:
def naive_bayes_predict(review, logprior, loglikelihood):
    word_l = process_review(review)

    p = logprior

    for word in word_l:
        if word in loglikelihood:
            p += loglikelihood[word]

    return p

In [ ]:
if naive_bayes_predict("""When you look at the first two Tom Holland-starrer Spider-Man movies, they had a breezy feel similar to the character’s emotional maturity. """,
                    logprior, loglikelihood) > 0:
  print ('позитивний')
else:
  print ('негативний')

позитивний


In [ ]:
if naive_bayes_predict("""This saccharine, reverential biopic about controversial music legend Michael Jackson is set to be one of the worst films of 2026 – removing "everything that might be deemed dramatic". It's bad. It's bad.""",
                    logprior, loglikelihood) > 0:
  print ('позитивний')
else:
  print ('негативний')

негативний


***6.Оцінити точність класифікатора на тестовому наборі даних***

Алгоритм:
- Для кожної рецензії в тестовому наборі:
    - Класифікувати рецензію з допомогою naive_bayes_predict.
    - Якщо ймовірність > 0, присвоїти рецензії мітку 1, інакше 0.
    - Додати передбачену мітку до списку.
- Порахувати середню абсолютну похибку між передбаченими та справжніми мітками.  
- Обчислити точність як 1 - похибка.

In [ ]:
def test_naive_bayes(test_x, test_y, logprior, loglikelihood):
    accuracy = 0

    y_hats = []
    for review in test_x:
        if naive_bayes_predict(review, logprior, loglikelihood) > 0:
            y_hat_i = 1
        else:
            y_hat_i = 0
        y_hats.append(y_hat_i)

    error = np.mean(np.absolute(y_hats - test_y))
    accuracy = 1 - error

    return accuracy

In [ ]:
test_naive_bayes(test_x, test_y, logprior, loglikelihood)

np.float64(0.795)

***7. Проаналізувати найбільш позитивні та негативні слова за співвідношенням їх частотностей***


In [ ]:
def lookup(freqs, word, label):
  "Пошук  скільки разів слово зустрілось у рецензіях певного класу"
  n = 0
  pair = (word, label)
  if pair in freqs:
    n = freqs[pair]

  return n

In [ ]:
def get_ratio(freqs, word):
    """Відношення частотності слова в позитивному класі до негативного."""
    pos_neg_ratio = {'positive': 0, 'negative': 0, 'ratio': 0.0}

    pos_neg_ratio['positive'] = freqs.get((word, 1), 0)
    pos_neg_ratio['negative'] = freqs.get((word, 0), 0)
    pos_neg_ratio['ratio'] = (pos_neg_ratio['positive'] + 1) / (pos_neg_ratio['negative'] + 1)

    return pos_neg_ratio

In [ ]:
def get_words_by_threshold(freqs, label, threshold, min_count=20):
    """Слова, чиє відношення частотностей >= threshold (label=1) або <= threshold (label=0)."""
    word_list = {}

    for word in {w for w, _ in freqs}:
        pos_neg_ratio = get_ratio(freqs, word)

        if pos_neg_ratio['positive'] + pos_neg_ratio['negative'] < min_count:
            continue

        if label == 1 and pos_neg_ratio['ratio'] >= threshold:
            word_list[word] = pos_neg_ratio
        elif label == 0 and pos_neg_ratio['ratio'] <= threshold:
            word_list[word] = pos_neg_ratio

    return word_list

In [ ]:
pos_words = get_words_by_threshold(freqs, 1, 3)

print("Найпозитивніші слова:")
for w, r in sorted(pos_words.items(), key=lambda x: x[1]['ratio'], reverse=True)[:15]:
    print(w, r)


Найпозитивніші слова:
shrek {'positive': 59, 'negative': 0, 'ratio': 60.0}
mulan {'positive': 78, 'negative': 1, 'ratio': 39.5}
gattaca {'positive': 36, 'negative': 0, 'ratio': 37.0}
guido {'positive': 31, 'negative': 0, 'ratio': 32.0}
flynt {'positive': 62, 'negative': 1, 'ratio': 31.5}
ordel {'positive': 30, 'negative': 0, 'ratio': 31.0}
leila {'positive': 29, 'negative': 0, 'ratio': 30.0}
sweetback {'positive': 27, 'negative': 0, 'ratio': 28.0}
taran {'positive': 26, 'negative': 0, 'ratio': 27.0}
homer {'positive': 26, 'negative': 0, 'ratio': 27.0}
mallori {'positive': 24, 'negative': 0, 'ratio': 25.0}
donkey {'positive': 22, 'negative': 0, 'ratio': 23.0}
rounder {'positive': 22, 'negative': 0, 'ratio': 23.0}
gile {'positive': 21, 'negative': 0, 'ratio': 22.0}
argento {'positive': 21, 'negative': 0, 'ratio': 22.0}


In [ ]:
neg_words = get_words_by_threshold(freqs, 0, 1/3)

print("\nНайнегативніші слова:")
for w, r in sorted(neg_words.items(), key=lambda x: x[1]['ratio'])[:15]:
    print(w, r)


Найнегативніші слова:
nbsp {'positive': 0, 'negative': 58, 'ratio': 0.01694915254237288}
psychlo {'positive': 0, 'negative': 35, 'ratio': 0.027777777777777776}
seagal {'positive': 0, 'negative': 30, 'ratio': 0.03225806451612903}
joli {'positive': 0, 'negative': 30, 'ratio': 0.03225806451612903}
brenner {'positive': 0, 'negative': 22, 'ratio': 0.043478260869565216}
tomb {'positive': 0, 'negative': 20, 'ratio': 0.047619047619047616}
pokemon {'positive': 0, 'negative': 20, 'ratio': 0.047619047619047616}
atroci {'positive': 0, 'negative': 20, 'ratio': 0.047619047619047616}
werewolf {'positive': 1, 'negative': 28, 'ratio': 0.06896551724137931}
schumach {'positive': 2, 'negative': 42, 'ratio': 0.06976744186046512}
8mm {'positive': 2, 'negative': 40, 'ratio': 0.07317073170731707}
jakob {'positive': 1, 'negative': 25, 'ratio': 0.07692307692307693}
magoo {'positive': 1, 'negative': 25, 'ratio': 0.07692307692307693}
jawbreak {'positive': 1, 'negative': 24, 'ratio': 0.08}
hudson {'positive': 1, 

***8. Проаналізувати помилки класифікації***

Алгоритм:
- Для кожної пари (рецензія, справжня мітка) з тестового набору:
    - Класифікувати рецензію та отримати передбачену ймовірність.
    - Якщо справжня мітка не збігається з передбаченою:
        - Вивести справжню мітку, передбачену мітку і оброблену рецензію.


In [ ]:
print('Truth Predicted Tweet')

errors = []
for x, y in zip(test_x, test_y):
    score = naive_bayes_predict(x, logprior, loglikelihood)
    y_hat = 1 if score > 0 else 0

    if y != y_hat:
        errors.append((x, y, y_hat, score))
        print('%d\t%d\t%s' % (y, y_hat, ' '.join(process_review(x)[:25])))

print(f'\nПомилок: {len(errors)} із {len(test_x)}')

Truth Predicted Tweet
1	0	think would easi spoof star trek tri spoof without cruelli mock that' bit difficult farfetch sci-fi seri enorm popular gigant fan base ongo instal televis
1	0	review there' someth mari said someth effect laughter everyth need make brief addendum postul laughter everyth unless watch jim abraham film see spoof easi love
1	0	showgirl second major outing product team pual verhoeven director joe eszterha writer last effort basic instinct play like sumptuou classic hitchcock dash raw sex sensual
1	0	one biggest clich serial killer film also one believ know one detect look wall pictur polic inform suddenli spot clue reveal killer sure use reason
1	0	strang day chronicl last two day lo angel local gear new millenium lenni nero ralph fienn goe busi peddl erot memori clip pine ex-girlfriend faith
1	0	matter time dilbert-esqu offic comedi hit screen howev rather come dilbert creator scott adam offic space instead base work anoth anim mike judg creator beavi
1	0	razor bla

Алгоритм:
- Для перших кількох помилок:
    - Обчислити внесок кожного слова в score (loglikelihood слова × кількість його повторів у рецензії).
    - Вивести слова з найбільшим за модулем внеском.
    - Пояснити причину помилки (сарказм, заперечення, змішана оцінка тощо).

In [ ]:
def top_words(review, n=5):
    """Практична демонстрація того, чому модель помилилась"""
    words = process_review(review)
    contrib = {w: loglikelihood[w] * words.count(w) for w in set(words) if w in loglikelihood}
    return sorted(contrib.items(), key=lambda x: abs(x[1]), reverse=True)[:n]

for x, y, y_hat, score in errors[:5]:
    print(f'Істина: {int(y)} | Передбачено: {y_hat} | Score: {score:.2f}')
    print(x[:300].replace('\n', ' '), '...')
    print('Найвпливовіші слова:', top_words(x))
    print('-' * 80)

Істина: 1 | Передбачено: 0 | Score: -7.06
you'd think it would be easy to spoof star trek , but try spoofing it without cruelly mocking it .  that's a bit more difficult .  the farfetched sci-fi series is , after all , enormously popular , what with its gigantic fan base and ongoing installments both on television and in movies .  if you're ...
Найвпливовіші слова: [('trek', np.float64(4.648885133171163)), ('nesmith', np.float64(-3.9879122594594563)), ('gwen', np.float64(-2.5023305441303165)), ('thermian', np.float64(-2.40609512000011)), ('sarri', np.float64(-2.3927473556756738))]
--------------------------------------------------------------------------------
Істина: 1 | Передбачено: 0 | Score: -17.53
in my review of there's something about mary , i said something to the effect of " laughter isn't everything . "  i now need to make a brief addendum to my postulate : laughter isn't everything , unless i'm watching a jim abrahams film .  see , spoofs are easy with me .  i love movies , 

***9. Протестувати класифікатор на власній рецензії***

In [ ]:
my_review1 = """To say that The Love Hypothesis is my most anticipated movie in recent memory feels like an understatement. I fell in love with Ali Hazelwood’s novel of the same name as soon as it was released back in 2021, and of course I’ve loved the couple that inspired the dynamic between Olive and Adam for even longer than that (shout out to my Reylos). As a fan of the book, the stars, the concept that inspired it, and definitely rom-coms in general, I’d consider myself squarely in the middle of whatever Venn diagram of target audiences The Love Hypothesis is going for. So did I like it?
In short: holy shit, yes. The Love Hypothesis delivered in every conceivable way. It is a faithful adaptation crafted with so much love, that is full of heart and is laugh out loud funny.
For those unfamiliar with the plot of Hazelwood’s debut novel, the story follows Olive Smith, a doctoral candidate who begins a fake relationship with Dr. Adam Carlsen in order to convince her best friend Anh that she is very much over the man she used to have a crush on, allowing Anh to date him guilt-free. Since Adam also has professional troubles that would be very much helped by appearing to have a serious girlfriend, he agrees. Hijinks and feelings ensue.
In case you missed her in the criminally-underrated Look Both Ways, then The Love Hypothesis is proof that if anyone can deliver on rom-com energy, it is Lili Reinhart. She excels at capturing every facet of Olive’s character, and together with a script from Sarah Rothschild and Ali Hazelwood, and direction from Claire Scanlon, brings out a lot of internal elements to the character that might have been lost on screen in the hands of another actor. The way she balances Olive’s growing feelings for Adam, and the deep love she already has for Anh, and how both of these impact her own work, make the character feel truly well-rounded.
And then there is her co-star Tom Bateman, whose casting was met with such a hilarious reaction in certain circles, I am frankly still chasing that high. His Adam is awkward and charming, with just enough menace to explain who students and peers alike have given him the nickname Dr. Evil. I would also like to take a minute to shout out Scanlon for two specific shots of Adam which, when paired with the threatening music, made me feel like all he was missing in that moment is a black helmet and a lightsaber with a crackly crossguard.
As the heart of the story, Reinhart and Bateman’s chemistry was always going to be the selling point, and luckily for us they soft-launched that last summer with a series of TikToks filmed during production, that they picked back up during the press tour. Luckily, though was there ever any doubt, their chemistry extends to the movie as well, as they go through the awkwardness of pretending to be deeply in love while still feeling mildly antagonistic, all the way through to developing genuine feelings for each other.
In terms of adaptational changes, there are surprisingly few. The moments and the lines that mean so much to readers like myself are preserved, and at times it truly feels like they just shook the book and the movie fell out. Are there some changes? Yes, of course. It’s a hop across mediums, and a lack of internal narration means a lot of things now need to be said out loud. It’s not the book shot for shot of course, there are some scenes changed or added for the flow of the movie, but it’s all in support of preserving the heart of the story, which it does in a beautiful way. I can honestly say that I wish all authors, especially the ones as incredible as Ali, such faithful, thoughtful adaptations of their work.
One moment in particular that I want to shout out, as I did in my review of the book five years ago (I cannot believe it’s been that long either) is the mention of Olive’s demisexuality. I said at the time how much it meant to me, as a demi person, to have that term explained twice, if not said by name outright, in a romance novel. I’ll admit, I didn’t expect that element of the character to make the jump to the screen. Another writer might have seen it as unnecessary, or seen it as slowing down the plot. But not only is it kept here, it’s actually named. I don’t mind saying that I gasped out loud. I can’t say how widespread this movie is going to become, but if it gets a few more people familiar with the term demisexual, then that’s a win in my book.
I hope The Love Hypothesis does well enough that it tells the powers that be that the people long for a rom-com. And not just one that will be swept under the rug of “see, no one wants romance!” One that is given the respect, and the investment, and the press tour that so many other genres seem to get. Prime, at least, has been pretty good at seeing the gap in the market and filling it, but as a romance lover, I can only hope this is the start of a much wider trend. I already know I’m going to be watching this movie several times over. Everyone, from the leads, to the supporting cast, to the comedic relief, brought their A-game. The team set out to make a classic, and I would say they’ve definitely succeeded.
"""
p = naive_bayes_predict(my_review1, logprior, loglikelihood)
print(p)

if p > 0:
    print('позитивний')
else:
    print('негативний')

print(top_words(my_review1))

15.78799009509559
позитивний
[('bateman', np.float64(4.921028172259181)), ('anh', np.float64(3.845577269363835)), ('adam', np.float64(-3.8264025056999778)), ('love', np.float64(3.032644745776345)), ('oliv', np.float64(2.3682128972160577))]


In [ ]:
my_review2 = """The Devil Wears Prada 2 is a dull, lifeless sequel that has no reason to exist. The original was sharp, funny and stylish, but this follow-up is boring from the first scene to the last. The script is lazy and full of tired jokes that fall flat, and the plot is so predictable that you can guess every twist twenty minutes in.

The worst part is how little the film does with its characters. Miranda Priestly, once a terrifying and brilliant villain, is reduced to a weak caricature who repeats the same cold one-liners. Andy and Emily feel pointless, and their scenes are awkward, forced and painfully unfunny. Even a great cast cannot save such a poor screenplay. The talented actors look bored, and honestly, I do not blame them.

The fashion, which should be the main attraction, is bland and disappointing. Instead of bold style we get endless product placement and cheap nostalgia. The film keeps winking at the audience, hoping we will forget how bad the story is. It is a shallow, cynical cash grab, and a terrible waste of a beloved classic.

At nearly two hours, it feels endless. I wanted to leave halfway through. Skip this disappointing mess and watch the original again instead. Awful, tedious and completely unnecessary."""
p = naive_bayes_predict(my_review2, logprior, loglikelihood)
print(p)

if p > 0:
    print('позитивний')
else:
    print('негативний')

print(top_words(my_review2))

-34.02786917885063
негативний
[('bore', np.float64(-2.5967154796087595)), ('miranda', np.float64(2.0927893060042724)), ('wast', np.float64(-1.7836065217807215)), ('worst', np.float64(-1.7540796042440867)), ('unfunni', np.float64(-1.657783717115004))]


Класифікувати неоднозначні тексти, де позитивні та негативні оцінки співіснують, досить складно, бо наївний баєсів класифікатор не враховує контекст і загальний тон автора. Через це доводиться вводити додаткові покращення: змінювати поріг класифікації (замість 0 використовувати інше значення - експериментально шукати), розширювати попередню обробку або застосовувати складніші моделі.

Один із прикладів неоднозначного тексту є:

In [ ]:
my_review3 = """“One Night Only,” director-producer Will Gluck‘s latest romantic comedy, is everything that’s troubling about the current conservative capitalist hellscape that is the United States of America writ large. Set three years after the U.S. Congress has enacted a mandate that extra-marital sex is only legal one night a year (August 7th, though why that date is never explained), the film centers on two hapless millennials—Allie (Monica Barbaro) and Owen (Callum Turner)—who are looking for more than just a hook-up while every other sex-crazed single around them goes at it like rabbits (Gluck’s limp idea of a city gone sex-crazed is less Hieronymus Bosch fever dream than it is ersatz Skinamax.)

Allie is a shy singer who makes a living belting out jingles for pharmaceutical products that sound like pop songs. Owen owns his own pizza joint. She’s on her own after her best friend (King Princess) meets a literal prince while in line for the bathroom, while Owen’s girlfriend (Maya Hawke, stuck in a thankless nothing of a role) says she wants to sleep with someone else this year. The two will, of course, meet-cute, over and over and over, as the night progresses.

Unfortunately, Travis Braun’s script is nothing but an edgy premise—think Jac Schaeffer’s “TiMER” (2009) mixed with “The Purge” (2013)—that exists solely to take milquetoast characters from one slapstick set up to another. It makes nods to the very real horrors of today, like tech bro-backed surveillance capitalism, rampant Big Pharma advertisements, and fascist police and politicians controlling our bodies, without ever taking the time to interrogate any of the sources that have led to our current political climate.

Good satire is rooted in the understanding and interrogation of systems, of the root causes of its satirical target. This can’t be achieved through quippy, outdated pop culture zingers and a handful of references to the country’s partisan political tension and “single issue voters,” all of which have the sole aim of making you laugh but don’t bother to make you think. Political satire cannot stand up if it doesn’t have a spine.

It’s a film all about sex that has no sense of eros between its central star-crossed would-be lovers, and only features the barest of nudity (the buttocks of a few different men running naked). That’s progress, I guess, since an earlier iteration of this film made in the ’80s or ’90s would have just been a flurry of tit shots. (Though apparently, we are still cursed with a woman forced to wear 6” stiletto heels for twelve hours as if her feet wouldn’t be bleeding and broken by dawn.)

Romance, too, is depicted in deeply conservative, heteronormative terms—casual sex is looked down on, open relationships only lead to guilt, etc. Oh, but don’t you worry, it’s got plenty of allegedly sex-positive product placement for Durex condoms and KY Jelly (as well as for CitiBikes and Dunkin, because what’s New York City without its big corporations clogging the streets).

Sex workers are thrown under the bus, shown only as thieves who fleece men in the woods at night, while queer sexuality is relegated to a few scenes with Allie’s roommate (a woefully underused Quintessa Swindell) and a handful of pans past blurry same-sex extras making out in the background. “This night is for the straights,” Swindell’s character says to Allie, before saying that queer people have worked around straight laws for centuries.

This is supposed to be a nod to queer resilience, but largely comes off as straightwashing the history of anti-LGBTQ legislation in this country that led to raids of queer clubs and incarceration of queer people in the not-too-distant past (let alone the current barrage of new anti-LGBTQ legislation currently in the works and on the books). It’s also ignorant of the long and fraught fight for the normalization of age-of-consent laws, which, by the way, did not happen in all states in this very country until 2003. Yes, that’s not even thirty years ago.

By the end, our intrepid protagonists have not had sex with anyone, because this is a film about Romance with a Capital R (and I guess sex is not part of romance!) and for that very same reason they ultimately embrace the fascist state of things. They aren’t the only ones who refuse to fight back. No one in this cursed film is protesting this mandate beyond “repeal” signs in various windows and some creative graffiti (like Lady Liberty in a chastity belt). The few activists we do see—one of Allie’s dates who’s on parole for fucking fourteen times in an eight-day period and an old man who refuses to wear love gloves—are used for jokes, and then are thrown into the clink and resolutely forgotten.

And perhaps that’s the point of the movie; that we’re already this deep into a fascist state of emergency, yet only a few people can actually be bothered to do anything about it beyond putting signs in their window. But the satire needs to be sharper (and smarter) for the movie not to fold in on itself. And if we are supposed to believe that the film is making some subversive statement about the state of sex and bodily autonomy in this country, it cannot end with its leads embracing their chains. That’s just warmed-over conservatism running around in sheep’s clothing.

You might ask, does a rom-com fail just because it refuses to actually engage with its political premise? Maybe not. But if the central romance falls flat, the jokes aren’t funny, and the desperate celebrity cameos grate like nails on a chalkboard, it does fail even at being “just a rom-com.”

In one sequence where Owen and Allie can’t manage to pull $200 out of a bodega ATM in order to buy the lone condom left in all of Manhattan, one of them finds a postcard with Deborah Kerr and Cary Grant in “An Affair To Remember” (1957) on it that reads: New York is For Lovers. One can only hope that the ghosts of Kerr, Grant, Leo McCarey, and hell, even Nora Ephron, haunt the makers of this putrid, regressive film that thinks justifying embracing fascism is okay if it’s in the name of love.

If this is what it takes to get a rom-com greenlit these days, maybe we should just let them die. Death with dignity is surely a better fate for the genre than fascist capitulation.
"""
p = naive_bayes_predict(my_review3, logprior, loglikelihood)
print(p)

if p > 10:
    print('позитивний')
else:
    print('негативний')

print(top_words(my_review3))


9.88684430875582
негативний
[('polit', np.float64(4.593310788653692)), ('fascist', np.float64(3.97670806934466)), ('conserv', np.float64(3.1390823244794497)), ('kerr', np.float64(3.0100052822043075)), ('sex-craz', np.float64(-2.9814592649036697))]


In [ ]:
import json

data = {
    'logprior': float(logprior),
    'loglikelihood': {w: float(v) for w, v in loglikelihood.items()},
    'stopwords': stopwords.words('english')
}

with open('naive_bayes_model.json', 'w') as outfile:
    json.dump(data, outfile)

from google.colab import files
files.download('naive_bayes_model.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>